### Question - 002 : Find the count of total matches played and won by each team using pyspark

In this video, we solve a PySpark interview question on how to calculate:

✅ The total number of matches played by each team

✅ The total number of matches won by each team

✅ An optimized PySpark approach using groupBy(), union(), and aggregation functions

This question is frequently asked in Data Engineer interviews at companies like Accenture, TCS, Infosys, and more!


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark

In [2]:
data = [
    ("India", "Australia", "India"),
    ("England", "Pakistan", "England"),
    ("South Africa", "New Zealand", "New Zealand"),
    ("West Indies", "Sri Lanka", "Sri Lanka"),
    ("India", "Pakistan", "India"),
    ("Australia", "England", "Australia"),
    ("New Zealand", "Sri Lanka", "New Zealand"),
    ("South Africa", "West Indies", "South Africa"),
    ("India", "England", "India"),
    ("Pakistan", "Australia", "Pakistan"),
]

columns = ["team_1", "team_2", "winner"]

df = spark.createDataFrame(data=data, schema=columns)

df.show(truncate=False)

+------------+-----------+------------+
|team_1      |team_2     |winner      |
+------------+-----------+------------+
|India       |Australia  |India       |
|England     |Pakistan   |England     |
|South Africa|New Zealand|New Zealand |
|West Indies |Sri Lanka  |Sri Lanka   |
|India       |Pakistan   |India       |
|Australia   |England    |Australia   |
|New Zealand |Sri Lanka  |New Zealand |
|South Africa|West Indies|South Africa|
|India       |England    |India       |
|Pakistan    |Australia  |Pakistan    |
+------------+-----------+------------+



#### The total number of matches played by each team

In [3]:
from pyspark.sql.functions import *
from pyspark.sql.window import *


matches_df = df.select(col("team_1").alias("team"), col("winner")) \
                .union( \
                    df.select(col("team_2").alias("team"), col("winner")) \
                ) \
                .withColumn("winner_flag", when(col("team") == col("winner"), 1).otherwise(0)) 
            

matches_df.show(truncate=False)


+------------+------------+-----------+
|team        |winner      |winner_flag|
+------------+------------+-----------+
|India       |India       |1          |
|England     |England     |1          |
|South Africa|New Zealand |0          |
|West Indies |Sri Lanka   |0          |
|India       |India       |1          |
|Australia   |Australia   |1          |
|New Zealand |New Zealand |1          |
|South Africa|South Africa|1          |
|India       |India       |1          |
|Pakistan    |Pakistan    |1          |
|Australia   |India       |0          |
|Pakistan    |England     |0          |
|New Zealand |New Zealand |1          |
|Sri Lanka   |Sri Lanka   |1          |
|Pakistan    |India       |0          |
|England     |Australia   |0          |
|Sri Lanka   |New Zealand |0          |
|West Indies |South Africa|0          |
|England     |India       |0          |
|Australia   |Pakistan    |0          |
+------------+------------+-----------+



In [4]:

result_df = matches_df.groupBy("team").agg(
    count("team").alias("total_match_played"),
    sum("winner_flag").alias("total_match_won")
)

result_df.show()

+------------+------------------+---------------+
|        team|total_match_played|total_match_won|
+------------+------------------+---------------+
|       India|                 3|              3|
|     England|                 3|              1|
|South Africa|                 2|              1|
| West Indies|                 2|              0|
|   Australia|                 3|              1|
| New Zealand|                 2|              2|
|    Pakistan|                 3|              1|
|   Sri Lanka|                 2|              1|
+------------+------------------+---------------+

